# Notes Bruin - Module 5 DE Zoomcamp

## Qu'est-ce que Bruin ?

**Bruin** = Outil open-source de data pipeline **tout-en-un**, écrit en Go. Il remplace un stack de 4 outils distincts par un seul framework unifié.

**Le problème du stack classique :**
```
Fivetran (ingestion) → dbt (transformation) → Great Expectations (qualité) → Airflow (orchestration)
         ↑                    ↑                         ↑                          ↑
     outil 1              outil 2                   outil 3                    outil 4
                 → quand ça casse : 4 dashboards à inspecter
```

**Avec Bruin :**
```
Ingestion → Transformation SQL/Python → Qualité → Orchestration
            ↑_____________ tout ça dans Bruin _______________↑
```

**Principes de design :**
- Tout est du code versionnable (pas de config en base de données / UI)
- Multi-langages : SQL + Python + R dans la même pipeline
- Qualité des données = citoyen de première classe, pas un afterthought
- Local-first : développe sur DuckDB, déploie sur BigQuery/Snowflake

---

## Architecture d'un projet Bruin

### Structure des fichiers

```
mon-projet/
├── .bruin.yml                  # Connexions + environnements (⚠️ TOUJOURS dans .gitignore)
└── pipeline/
    ├── pipeline.yml            # Nom, schedule, variables, connexions par défaut
    └── assets/
        ├── ingestion/
        │   ├── trips.py        # Asset Python d'ingestion
        │   ├── requirements.txt
        │   ├── payment_lookup.asset.yml   # Asset seed (CSV → table)
        │   └── payment_lookup.csv
        ├── staging/
        │   └── trips.sql       # Asset SQL de nettoyage/déduplication
        └── reports/
            └── trips_report.sql  # Asset SQL d'agrégation
```

### Les 3 fichiers clés

**`.bruin.yml`**
- Définit les environnements (`default`, `production`)
- Contient les credentials de connexion (DuckDB, BigQuery, Snowflake...)
- **⚠️ OBLIGATOIREMENT dans `.gitignore`** — contient des secrets

```yaml
default_environment: default
environments:
    default:
        connections:
            duckdb:
                - name: "duckdb-default"
                  path: "./duckdb.db"
```

**`pipeline.yml`**
- `name` : identifiant de la pipeline (visible dans les logs + `BRUIN_PIPELINE`)
- `schedule` : fréquence d'exécution (`daily`, `hourly`, `weekly`, cron)
- `start_date` : date la plus ancienne pour les backfills
- `default_connections` : connexion par défaut par type de plateforme
- `variables` : variables avec JSON Schema et valeurs par défaut

```yaml
name: nyc-taxi
schedule: "daily"
start_date: "2022-01-01"
default_connections:
  duckdb: duckdb-default
variables:
  taxi_types:
    type: array
    items:
      type: string
    default: ["yellow"]
```

**`assets/`**
- Contient tous les scripts Python, SQL, YAML de la pipeline
- Organisation recommandée : `ingestion/` → `staging/` → `reports/`

---

## Les 4 types d'assets

### 1. Asset Python (`.py`)
Ingestion de données depuis des sources externes. Retourne un DataFrame que Bruin matérialise.

```python
"""@bruin
name: ingestion.trips
type: python
image: python:3.11
connection: duckdb-default
materialization:
  type: table
  strategy: append
@bruin"""

def materialize():
    # Bruin injecte automatiquement ces variables d'environnement
    start_date = os.environ["BRUIN_START_DATE"]   # YYYY-MM-DD
    end_date   = os.environ["BRUIN_END_DATE"]     # YYYY-MM-DD
    bruin_vars = json.loads(os.environ.get("BRUIN_VARS", "{}"))
    
    # ... fetch data ...
    return dataframe
```

### 2. Asset SQL (`.sql`)
Transformation, nettoyage, agrégation.

```sql
/* @bruin
name: staging.trips
type: duckdb.sql
depends:
  - ingestion.trips
materialization:
  type: table
  strategy: time_interval
  incremental_key: pickup_datetime
  time_granularity: timestamp
@bruin */

SELECT * FROM ingestion.trips
WHERE pickup_datetime >= '{{ start_datetime }}'
  AND pickup_datetime <  '{{ end_datetime }}'
```

### 3. Asset Seed (`.asset.yml` + `.csv`)
Charge un fichier CSV statique dans une table.

### 4. Asset Ingestr (`.asset.yml`)
Copie des données d'une source vers une destination sans coder.

---

## Stratégies de matérialisation

| Stratégie | Usage | Comportement |
|-----------|-------|--------------|
| `table` | Petites tables, full refresh | Drop + recrée toute la table |
| `view` | Données always-fresh | Vue SQL, pas de stockage |
| `append` | Ingestion brute, logs | INSERT sans toucher l'existant |
| `merge` | Upsert avec clé unique | UPDATE existant + INSERT nouveau |
| `time_interval` | Données time-series | DELETE fenêtre + INSERT fenêtre |
| `delete+insert` | Refresh par partitions | DELETE matching + INSERT |

**Pattern pour une pipeline ELT :**
```
Ingestion → append         (données brutes, doublons gérés downstream)
Staging   → time_interval  (re-process une période sans full refresh)
Reports   → time_interval  (même clé que staging pour la cohérence)
```

**⚠️ Règle time_interval :** Ton query DOIT filtrer sur `{{ start_datetime }}` / `{{ end_datetime }}`.
Sans ce filtre : Bruin supprime la fenêtre mais insert TOUT → doublons.

---

## Variables d'environnement Bruin

Dans les assets Python, Bruin injecte automatiquement :

| Variable | Format | Description |
|----------|--------|-------------|
| `BRUIN_START_DATE` | `YYYY-MM-DD` | Date de début du run |
| `BRUIN_END_DATE` | `YYYY-MM-DD` | Date de fin du run |
| `BRUIN_START_DATETIME` | ISO datetime | Datetime de début |
| `BRUIN_END_DATETIME` | ISO datetime | Datetime de fin |
| `BRUIN_VARS` | JSON string | Variables du pipeline |
| `BRUIN_PIPELINE` | string | Nom de la pipeline |

Dans les assets SQL, Bruin injecte via Jinja :
- `{{ start_datetime }}` / `{{ end_datetime }}`
- `{{ var.taxi_types }}`

---

## Qualité des données

### Checks intégrés (dans la définition des colonnes)

```yaml
columns:
  - name: pickup_datetime
    type: timestamp
    primary_key: true
    checks:
      - name: not_null
  - name: fare_amount
    type: float
    checks:
      - name: non_negative
  - name: vendor_id
    type: integer
    checks:
      - name: positive
```

**Checks disponibles :** `not_null`, `unique`, `positive`, `non_negative`, `accepted_values`

### Custom checks

```yaml
custom_checks:
  - name: row_count_positive
    description: "La table ne doit pas être vide"
    query: SELECT COUNT(*) > 0 FROM staging.trips
    value: 1
```

---

## Commandes CLI essentielles

| Commande | Usage |
|----------|-------|
| `bruin init <template> <folder>` | Initialise un projet depuis un template |
| `bruin validate <path>` | Vérifie syntaxe + dépendances sans exécuter (rapide !) |
| `bruin run <path>` | Execute pipeline ou asset |
| `bruin run --downstream` | Execute l'asset + tous ses descendants |
| `bruin run --full-refresh` | Recrée les tables from scratch |
| `bruin run --only checks` | Lance uniquement les quality checks |
| `bruin lineage <path>` | Affiche les dépendances (upstream/downstream) |
| `bruin query --connection <conn> --query "..."` | Requête ad-hoc |
| `bruin connections list` | Liste les connexions configurées |
| `bruin connections test --name <name>` | Teste une connexion |

**Flags de run :**
```bash
bruin run ./pipeline/pipeline.yml \
  --environment default \
  --start-date 2022-01-01 \
  --end-date 2022-01-31 \
  --full-refresh \
  --var 'taxi_types=["yellow"]'
```

---

## Pattern de déduplication SQL

Les données TLC n'ont pas d'ID unique — on crée une clé composite :

```sql
WITH deduplicated AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                pickup_datetime,
                dropoff_datetime,
                pickup_location_id,
                dropoff_location_id,
                fare_amount
            ORDER BY extracted_at DESC  -- Garde la version la plus récente
        ) AS rn
    FROM source_table
)
SELECT * FROM deduplicated WHERE rn = 1
```

---

## Tips & Bonnes pratiques

**Dev local — gestion du volume de données :**
```bash
# Pendant le dev : 1-3 mois max (les fichiers parquet font ~100MB chacun)
bruin run ./pipeline/pipeline.yml --start-date 2022-01-01 --end-date 2022-03-01

# Full backfill une fois la pipeline validée
bruin run ./pipeline/pipeline.yml --start-date 2019-01-01 --end-date 2025-11-30
```

**Ordre de développement recommandé :**
1. `bruin validate` → toujours avant de `run`
2. `--full-refresh` → obligatoire au premier run (la table n'existe pas encore)
3. Tester asset par asset avant de lancer la pipeline complète
4. `bruin query` pour vérifier les résultats après chaque run

**`.gitignore` minimal pour un projet Bruin :**
```
.bruin.yml      # Credentials — JAMAIS commit
duckdb.db       # Fichier de base locale — peut faire 100MB+
*.db
```

---

## Erreurs rencontrées

### `dlopen() failed: libduckdb.so: No such file or directory`
Bruin a créé le fichier de config mais n'a pas téléchargé le driver DuckDB.

**Réflexe général :** `dlopen() failed` + `No such file or directory` = librairie système manquante.
Vérifier d'abord avec `ls -la <chemin>` si le fichier existe vraiment.

**Fix :**
```bash
mkdir -p /home/kalou/.config/adbc/drivers/duckdb_linux_amd64_v1.4.4
cd /home/kalou/.config/adbc/drivers/duckdb_linux_amd64_v1.4.4
wget https://github.com/duckdb/duckdb/releases/download/v1.4.4/libduckdb-linux-amd64.zip
unzip libduckdb-linux-amd64.zip
```

⚠️ **PATTERN À RETENIR EN DE** : ce type d'erreur se recroise souvent avec les librairies C/Go.

### `Table with name trips does not exist`
La stratégie `time_interval` tente un DELETE avant de créer la table.

**Fix :** Toujours utiliser `--full-refresh` au premier run d'un asset.

### `there's no secret with the name 'duckdb-default'`
La connexion n'est pas configurée dans `.bruin.yml`.

**Fix :** Ajouter la connexion dans `.bruin.yml` :
```yaml
environments:
    default:
        connections:
            duckdb:
                - name: "duckdb-default"
                  path: "./duckdb.db"
```

---

## Architecture ELT complète

```
Source externe (TLC API)
        ↓
[ingestion.trips]         Python asset — append — données brutes
[ingestion.payment_lookup] Seed asset — CSV statique
        ↓
[staging.trips]           SQL asset — time_interval — nettoyage + dédup + enrichissement
        ↓
[reports.trips_report]    SQL asset — time_interval — agrégations analytics
        ↓
Dashboard / BI Tool
```

---

*Notes du Module 5 - DataTalks Club DE Zoomcamp*